In [ ]:
import boto3
import time
import logging
from datetime import datetime, date, timedelta, timezone
from typing import List, Optional, Tuple, Dict, Any
from decimal import Decimal

from botocore.exceptions import ClientError
from pyspark.sql import functions as F
from pyspark.sql import Row

In [ ]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("AWSCostExplorer")

logging.getLogger("boto3").setLevel(logging.WARNING)
logging.getLogger("botocore").setLevel(logging.WARNING)

In [ ]:
dbutils.widgets.text("catalog", "", "CATALOG")
dbutils.widgets.text("schema", "", "SCHEMA")
dbutils.widgets.text("overlap_days", "3", "Overlap days (min 2)")

In [ ]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
overlap_days = int(dbutils.widgets.get("overlap_days") or "3")

if overlap_days < 2:
    logger.warning("overlap_days < 2; forcing to 2 for cost convergence best practice.")
    overlap_days = 2

audit_table = f"{catalog}.{schema}.dbspend360_audit_log"
target_table = f"{catalog}.{schema}.dbspend360_cloud_cost_explorer"
error_log_table = f"{catalog}.{schema}.dbspend360_error_log"
breakdown_table = f"{catalog}.{schema}.dbspend360_other_cost_breakdown"

In [ ]:
# =======================================================
# AWS Cost Classification Framework
# =======================================================

# Single source of truth for AWS service-to-category mapping.
# Add new services here to classify them. Any service NOT in this
# map is routed to "other" (never silently assigned to compute).
AWS_SERVICE_CATEGORIES: Dict[str, str] = {
    "Amazon Elastic Compute Cloud - Compute": "compute",
    "EC2 - Other": "compute",
    "Amazon Elastic Block Store": "storage",
    "Amazon Simple Storage Service": "storage",
    "Elastic Load Balancing": "network",
    "AWS Data Transfer": "network",
    "Amazon Virtual Private Cloud": "network",
}

VALID_COST_CATEGORIES = {"compute", "storage", "network", "other"}


def build_aws_category_column():
    """Build a PySpark Column that classifies service_name using AWS_SERVICE_CATEGORIES.

    Uses F.create_map so the dict is the sole source of truth.
    Unknown services map to 'other' -- never silently to 'compute'.
    """
    from itertools import chain
    mapping_pairs = list(chain.from_iterable(AWS_SERVICE_CATEGORIES.items()))
    mapping_expr = F.create_map(*[F.lit(x) for x in mapping_pairs])
    return F.coalesce(mapping_expr[F.col("service_name")], F.lit("other"))


# =======================================================
# AWS Cost Client
# =======================================================


class AWSCostClient:
    """Client for querying AWS Cost Explorer API for Databricks cluster costs.

    Uses Databricks service credentials to assume an IAM role with
    ce:GetCostAndUsage permissions. The Cost Explorer API endpoint
    is only available in us-east-1, regardless of where resources run.

    Queries costs grouped by ClusterId tag AND SERVICE dimension,
    producing per-cluster daily costs segmented into compute, storage,
    and network categories.
    """

    CE_REGION = "us-east-1"

    DEFAULT_SERVICES = [
        "Amazon Elastic Compute Cloud - Compute",
        "EC2 - Other", #EC2 ancillary costs (Elastic IPs, NAT Gateway, data transfer)
        "Amazon Elastic Block Store",
        "Amazon Simple Storage Service",
        "Elastic Load Balancing",
        "AWS Data Transfer",
        "Amazon Virtual Private Cloud",
    ]

    MAX_CHUNK_DAYS = 30
    MAX_RETRIES = 5
    BASE_RETRY_DELAY = 5

    def __init__(self, service_credential_name: str = "dbspend-read-ce"):
        session = boto3.Session(
            botocore_session=dbutils.credentials.getServiceCredentialsProvider(
                service_credential_name
            ),
            region_name=self.CE_REGION,
        )
        self.client = session.client("ce")

    # -------- Public API --------

    def get_cluster_costs_daily(
        self,
        start_date: date,
        end_date: date,
        tag_key: str = "ClusterId",
        services: Optional[List[str]] = None,
        metric: str = "AmortizedCost",
    ):
        """Query CE for per-cluster daily costs with per-service detail.

        Uses dual GroupBy (TAG + SERVICE dimension) to get cost segmentation
        in a single API call. Returns per-service rows that the caller
        aggregates into compute/storage/network categories.

        Returns:
            Spark DataFrame with columns: cluster_id, service_name, cost,
            currency, cost_incurred_date — or None if no cost data found.
        """
        if services is None:
            services = self.DEFAULT_SERVICES

        chunks = self._build_chunks(start_date, end_date)
        all_rows: List[Dict[str, Any]] = []

        for i, (chunk_start, chunk_end) in enumerate(chunks):
            logger.info(f"Querying CE chunk {i+1}/{len(chunks)}: {chunk_start} → {chunk_end}")
            rows = self._query_with_retries(
                chunk_start, chunk_end, tag_key, services, metric
            )
            all_rows.extend(rows)
            if len(chunks) > 1:
                time.sleep(1)

        if not all_rows:
            return None

        return self._rows_to_spark_df(all_rows)

    # -------- Chunking --------

    def _build_chunks(self, start: date, end: date) -> List[Tuple[date, date]]:
        chunks = []
        current = start
        while current <= end:
            chunk_end = min(current + timedelta(days=self.MAX_CHUNK_DAYS - 1), end)
            chunks.append((current, chunk_end))
            current = chunk_end + timedelta(days=1)
        return chunks

    # -------- CE params construction --------

    def _build_ce_params(
        self,
        start: date,
        end: date,
        tag_key: str,
        services: List[str],
        metric: str,
    ) -> dict:
        """Build GetCostAndUsage request parameters.

        Uses dual GroupBy: TAG (ClusterId) + DIMENSION (SERVICE) so we
        get per-service costs per cluster in a single API call.
        CE TimePeriod.End is exclusive, so we add 1 day.
        """
        return {
            "TimePeriod": {
                "Start": start.isoformat(),
                "End": (end + timedelta(days=1)).isoformat(),
            },
            "Granularity": "DAILY",
            "Metrics": [metric],
            "GroupBy": [
                {"Type": "TAG", "Key": tag_key},
                {"Type": "DIMENSION", "Key": "SERVICE"},
            ],
            "Filter": {"Dimensions": {"Key": "SERVICE", "Values": services}},
        }

    # -------- Retry logic --------

    def _query_with_retries(
        self,
        start: date,
        end: date,
        tag_key: str,
        services: List[str],
        metric: str,
    ) -> List[Dict[str, Any]]:
        """Execute a CE query with exponential-backoff retries.

        Handles AWS CE rate limiting (LimitExceededException) with
        progressively longer delays, and retries transient errors.
        """
        params = self._build_ce_params(start, end, tag_key, services, metric)
        last_exception = None

        for attempt in range(self.MAX_RETRIES):
            try:
                return self._execute_paginated_query(params, metric)
            except ClientError as e:
                last_exception = e
                error_code = e.response["Error"]["Code"]

                if error_code == "LimitExceededException":
                    wait = min(self.BASE_RETRY_DELAY * (2 ** attempt), 120)
                    logger.warning(
                        f"Rate limited (attempt {attempt + 1}/{self.MAX_RETRIES}), "
                        f"waiting {wait}s"
                    )
                    time.sleep(wait)
                elif attempt < self.MAX_RETRIES - 1:
                    wait = 2 ** attempt
                    logger.warning(
                        f"ClientError {error_code} (attempt {attempt + 1}), "
                        f"retrying in {wait}s: {e}"
                    )
                    time.sleep(wait)
                else:
                    raise
            except Exception as e:
                last_exception = e
                if attempt < self.MAX_RETRIES - 1:
                    wait = 2 ** attempt
                    logger.warning(
                        f"Unexpected error (attempt {attempt + 1}), "
                        f"retrying in {wait}s: {e}"
                    )
                    time.sleep(wait)
                else:
                    raise

        raise last_exception

    # -------- Pagination --------

    def _execute_paginated_query(
        self, params: dict, metric: str
    ) -> List[Dict[str, Any]]:
        """Execute a CE query, following pagination tokens to completion."""
        rows: List[Dict[str, Any]] = []
        request_params = params.copy()

        while True:
            response = self.client.get_cost_and_usage(**request_params)
            rows.extend(self._parse_response(response, metric))

            next_token = response.get("NextPageToken")
            if not next_token:
                break

            request_params["NextPageToken"] = next_token
            time.sleep(0.5)

        return rows

    # -------- Response parsing --------

    def _parse_response(
        self, response: dict, metric: str
    ) -> List[Dict[str, Any]]:
        """Parse a single CE response page into flat row dicts.

        With dual GroupBy, each group has two Keys:
          keys[0] = "TagKey$TagValue" (e.g. "ClusterId$0101-abcdef")
          keys[1] = SERVICE dimension value (e.g. "Amazon Elastic Compute Cloud - Compute")
        """
        rows = []
        for time_block in response.get("ResultsByTime", []):
            period_date = time_block["TimePeriod"]["Start"]

            for group in time_block.get("Groups", []):
                keys = group.get("Keys", [])
                if len(keys) < 2:
                    continue

                raw_tag = keys[0]
                cluster_id = raw_tag.split("$")[-1] if "$" in raw_tag else raw_tag
                service_name = keys[1]

                metric_data = group["Metrics"].get(metric, {})
                amount = float(Decimal(metric_data.get("Amount", "0")))
                currency = metric_data.get("Unit", "USD")

                if amount == 0.0:
                    continue

                rows.append({
                    "cluster_id": cluster_id,
                    "service_name": service_name,
                    "cost": amount,
                    "currency": currency,
                    "cost_incurred_date": period_date,
                })

        return rows

    # -------- Spark conversion --------

    def _rows_to_spark_df(self, rows: List[Dict[str, Any]]):
        """Convert parsed rows to a Spark DataFrame with proper date types."""
        df = spark.createDataFrame(rows)
        return df.withColumn(
            "cost_incurred_date",
            F.to_date(F.col("cost_incurred_date"), "yyyy-MM-dd"),
        )

In [ ]:
# =======================================================
# APP
# =======================================================
class AWSCostReporterApp:
    """Orchestrates incremental AWS cost ingestion into the cloud cost table.

    Reads the audit log to determine the last successful run, queries
    AWS Cost Explorer for the incremental window (with overlap for
    idempotent MERGE), classifies costs into compute/storage/network/other,
    and upserts results into the target table.

    Invariant enforced: cloud_cost = compute_cost + storage_cost + network_cost + other_cost
    """

    def __init__(self):
        self.client = AWSCostClient()

    def run(self):
        start_dt, end_dt = self._get_date_window()

        logger.info(
            f"Querying AWS CE cost from {start_dt} to {end_dt} "
            f"(overlap_days={overlap_days})"
        )

        if start_dt > end_dt:
            message = (
                f"Invalid date window: start_dt={start_dt} > end_dt={end_dt}. "
                f"Check audit table and overlap_days={overlap_days}."
            )
            logger.error(message)
            self._log_run(start_dt, end_dt, "FAILED", 0, message)
            dbutils.notebook.exit("FAILED: Invalid date window.")

        self._ensure_schema_columns()

        spark_df = self.client.get_cluster_costs_daily(
            start_date=start_dt,
            end_date=end_dt,
        )

        quality_msg = f"overlap_days={overlap_days}"

        if spark_df is None or spark_df.limit(1).count() == 0:
            logger.info("No AWS cost data returned for the requested range.")
            merged_row_count = 0
        else:
            inc_df = (
                spark_df
                .filter(
                    (F.col("cluster_id").isNotNull()) &
                    (F.col("cluster_id") != "")
                )
                .filter(F.col("cost_incurred_date").isNotNull())
            )

            if inc_df.limit(1).count() == 0:
                logger.info("No rows after filtering by cluster_id and cost_incurred_date.")
                merged_row_count = 0
            else:
                self._log_unclassified_services(inc_df)
                self._write_other_cost_breakdown(inc_df)

                agg_df = self._classify_and_aggregate(inc_df)
                merged_row_count = agg_df.count()

                quality_msg = self._compute_quality_metrics(agg_df, merged_row_count)

                agg_df.createOrReplaceTempView("cloud_cost_inc")

                spark.sql(f"""
                MERGE INTO {target_table} AS t
                USING cloud_cost_inc AS s
                ON  t.cluster_id = s.cluster_id
                AND t.currency = s.currency
                AND t.cost_incurred_date = s.cost_incurred_date
                WHEN MATCHED THEN
                  UPDATE SET
                    t.cloud_cost      = s.cloud_cost,
                    t.compute_cost    = s.compute_cost,
                    t.storage_cost    = s.storage_cost,
                    t.network_cost    = s.network_cost,
                    t.other_cost      = s.other_cost,
                    t.updated_at      = current_timestamp()
                WHEN NOT MATCHED THEN
                  INSERT (cluster_id, cloud_cost, compute_cost, storage_cost, network_cost, other_cost,
                          currency, cost_incurred_date, created_at, updated_at)
                  VALUES (s.cluster_id, s.cloud_cost, s.compute_cost, s.storage_cost, s.network_cost, s.other_cost,
                          s.currency, s.cost_incurred_date,
                          current_timestamp(), current_timestamp())
                """)

        logger.info(
            f"Merged {merged_row_count} rows into {target_table} "
            f"for {start_dt} → {end_dt} (overlap_days={overlap_days})."
        )

        self._log_run(start_dt, end_dt, "SUCCESS", merged_row_count, quality_msg)

    def _ensure_schema_columns(self):
        """Add cost segmentation columns if they don't exist yet."""
        existing = {c.name for c in spark.table(target_table).schema}
        missing = [c for c in ("compute_cost", "storage_cost", "network_cost", "other_cost") if c not in existing]
        if missing:
            cols_sql = ", ".join(f"{c} DOUBLE" for c in missing)
            spark.sql(f"ALTER TABLE {target_table} ADD COLUMNS ({cols_sql})")
            logger.info(f"Added columns to {target_table}: {missing}")

    def _log_unclassified_services(self, df):
        """Log unclassified services to both logger and error_log table."""
        known = set(AWS_SERVICE_CATEGORIES.keys())
        svc_costs = (
            df.groupBy("service_name")
            .agg(
                F.sum("cost").alias("total_cost"),
                F.count("*").alias("row_count"),
            )
            .collect()
        )
        unknown_rows = [r for r in svc_costs if r.service_name not in known]

        if not unknown_rows:
            return

        svc_names = [r.service_name for r in unknown_rows]
        logger.warning(f"Unclassified AWS services (routed to other_cost): {svc_names}")

        try:
            from pyspark.sql.types import StructType, StructField, StringType, TimestampType

            error_schema = StructType([
                StructField("source_system", StringType()),
                StructField("error_type", StringType()),
                StructField("cluster_id", StringType()),
                StructField("job_id", StringType()),
                StructField("run_id", StringType()),
                StructField("usage_date", StringType()),
                StructField("currency", StringType()),
                StructField("error_detail", StringType()),
                StructField("raw_record", StringType()),
                StructField("created_at", TimestampType()),
            ])
            error_records = [
                Row(
                    source_system="AWS",
                    error_type="UNCLASSIFIED_COST",
                    cluster_id=None,
                    job_id=None,
                    run_id=None,
                    usage_date=None,
                    currency=None,
                    error_detail=f"Unclassified service: {r.service_name}, total_cost=${r.total_cost:.4f}, rows={r.row_count}",
                    raw_record=None,
                    created_at=datetime.now(timezone.utc),
                )
                for r in unknown_rows
            ]
            spark.createDataFrame(error_records, schema=error_schema).write.mode("append").insertInto(error_log_table)
        except Exception as e:
            logger.warning(f"Failed to write unclassified services to error_log: {e}")

    def _write_other_cost_breakdown(self, inc_df):
        """Write per-service detail for 'other' category costs to the breakdown table.

        Aggregates unclassified service costs by (date, cluster, service) and
        MERGEs into dbspend360_other_cost_breakdown for drilldown queries.
        Idempotent: reruns update existing rows via MERGE.
        """
        classified = inc_df.withColumn("category", build_aws_category_column())

        other_df = (
            classified
            .filter(F.col("category") == "other")
            .groupBy("cluster_id", "service_name", "currency", "cost_incurred_date")
            .agg(F.sum("cost").alias("cost"))
            .withColumn("source_system", F.lit("AWS"))
            .withColumn("created_at", F.current_timestamp())
            .withColumn("updated_at", F.current_timestamp())
        )

        if other_df.limit(1).count() == 0:
            logger.info("No 'other' category costs to write to breakdown table.")
            return

        other_df.createOrReplaceTempView("other_cost_breakdown_inc")

        spark.sql(f"""
        MERGE INTO {breakdown_table} AS t
        USING other_cost_breakdown_inc AS s
        ON  t.cost_incurred_date = s.cost_incurred_date
        AND t.cluster_id = s.cluster_id
        AND t.source_system = s.source_system
        AND t.service_name = s.service_name
        AND t.currency = s.currency
        WHEN MATCHED THEN
          UPDATE SET t.cost = s.cost, t.updated_at = current_timestamp()
        WHEN NOT MATCHED THEN
          INSERT (cost_incurred_date, cluster_id, source_system, service_name,
                  cost, currency, created_at, updated_at)
          VALUES (s.cost_incurred_date, s.cluster_id, s.source_system, s.service_name,
                  s.cost, s.currency, current_timestamp(), current_timestamp())
        """)

        breakdown_count = other_df.count()
        logger.info(f"Wrote {breakdown_count} other cost breakdown rows to {breakdown_table}")

    @staticmethod
    def _classify_and_aggregate(inc_df):
        """Classify per-service rows using the centralized AWS_SERVICE_CATEGORIES map.

        Uses build_aws_category_column() so the dict is the single source of truth.
        Unknown services are classified as 'other' -- never silently as 'compute'.

        Invariant: cloud_cost = compute_cost + storage_cost + network_cost + other_cost
        """
        classified = inc_df.withColumn("category", build_aws_category_column())

        return (
            classified
            .groupBy("cluster_id", "currency", "cost_incurred_date")
            .agg(
                F.sum(F.when(F.col("category") == "compute", F.col("cost")).otherwise(0)).alias("compute_cost"),
                F.sum(F.when(F.col("category") == "storage", F.col("cost")).otherwise(0)).alias("storage_cost"),
                F.sum(F.when(F.col("category") == "network", F.col("cost")).otherwise(0)).alias("network_cost"),
                F.sum(F.when(F.col("category") == "other", F.col("cost")).otherwise(0)).alias("other_cost"),
            )
            .withColumn("cloud_cost", F.col("compute_cost") + F.col("storage_cost") + F.col("network_cost") + F.col("other_cost"))
            .withColumn("created_at", F.current_timestamp())
            .withColumn("updated_at", F.current_timestamp())
        )

    @staticmethod
    def _compute_quality_metrics(agg_df, row_count):
        """Compute and log data quality / classification coverage metrics."""
        metrics = agg_df.agg(
            F.sum("cloud_cost").alias("total_cost"),
            F.sum("compute_cost").alias("classified_compute"),
            F.sum("storage_cost").alias("classified_storage"),
            F.sum("network_cost").alias("classified_network"),
            F.sum("other_cost").alias("unclassified_cost"),
        ).collect()[0]

        total = float(metrics.total_cost or 0)
        unclassified = float(metrics.unclassified_cost or 0)
        classified = total - unclassified
        coverage_pct = (classified / total * 100) if total > 0 else 100.0

        msg = (
            f"overlap_days={overlap_days}, rows={row_count}, "
            f"classification_coverage={coverage_pct:.1f}%, "
            f"classified_cost={classified:.2f}, "
            f"unclassified_cost={unclassified:.2f}, "
            f"total_cost={total:.2f}"
        )
        logger.info(f"Data quality: {msg}")
        return msg

    def _get_date_window(self) -> Tuple[date, date]:
        """Determine the incremental date window from the audit log.

        On first run (no prior SUCCESS entries), defaults to 365 days back.
        On subsequent runs, uses the last successful end_date minus
        overlap_days for idempotent re-MERGE coverage.
        """
        wm = (
            spark.table(audit_table)
                 .filter("table_name = 'dbspend360_cloud_cost_explorer' AND status = 'SUCCESS'")
        )

        if wm.limit(1).count() == 0:
            last_end_date = datetime.now(timezone.utc).date() - timedelta(
                days=365 - overlap_days
            )
        else:
            last_end_date = wm.agg(F.max("end_date")).collect()[0][0]

        start_dt = last_end_date - timedelta(days=overlap_days - 1)
        end_dt = datetime.now(timezone.utc).date()
        return start_dt, end_dt

    def _log_run(self, start_dt, end_dt, status, row_count, message=""):
        """Append a run record to the audit log table."""
        run_log_df = spark.createDataFrame([
            Row(
                table_name="dbspend360_cloud_cost_explorer",
                start_date=start_dt,
                end_date=end_dt,
                status=status,
                row_count=int(row_count),
                message=message,
                created_at=datetime.now(timezone.utc),
            )
        ])
        run_log_df.write.mode("append").insertInto(audit_table)

In [ ]:
# =======================================================
# Execute
# =======================================================
app = AWSCostReporterApp()
app.run()